# 9.2 超参数优化

**实践中，普适的超参数是存在的，搜索时一般在经验值中搜索就可以到很好的效果**

### 1. 确定超参数搜索空间
<img src="paste_image/2026-01-30-10-55-13.png" width="78%">



### 2. HPO 算法
- Black-box: 将每次训练任务当作黑盒，只观察和比较训练结果
- Multi-fidelity: 用近似的任务比较超参数之间的相对好坏
    - 减少数据：在 subsampled 数据集上训练
    - 简化模型：缩小模型尺寸（层数、通道），同一套超参数在相似结构的模型上表现大体相似
    - 设置早停措施

- 实际上使用较多的 HPO 算法：随机搜索、Hyperband

### Black-box Optimazition
**Grid search**：暴力穷举参数空间，开销指数级增加

**Random search**：从参数空间随机选择 config，重复 n 次，可以设置早停。很有效

**Bayesian Optimazition (BO)**：学习一个函数：从超参数映射到目标函数

- 每训练一次都可以得到一个数据点
- 会根据之前的实验来选择下一次训练的超参数
- Surrogate model：根据超参数拟合目标函数值的函数
    - 概率回归模型：随机森林、Gaussain process

    x轴为search space，y轴为目标函数值。黑色表示真实值，蓝色表示预测的概率分布，虚线表示预测的均值即预测值。

    <img src="paste_image/2026-01-30-12-12-50.png" width="85%">

- Acquisition function：评估下一个采样点
    - 值大表示 Surrogate model 对这块区域的预测不够置信，但是认为目标函数值可能比较高
    - Trade of exploration and exploitation

- Limnitation:
    - 初始阶段，几乎是随机搜索
    - 优化过程是 sequential 的，**不能并行**

### Multi-fidelity

**Successive Halving, SH**: 尽早淘汰不靠谱的超参数
1. 随机选择 n 组超参数，每组训练 m epoches（开始时一般n比较大10、100，m比较小1、2） 
2. 重复以下步骤，直到只剩一组超参数：
    - 保留精度最好的 n/2 组超参数，训练 2m epoches
- 根据训练开销选择 n 和 m：开销 = 训练任务(n) * 训练时间(m)，是一个定值
- 先确定最终模型的 epoches t，根据参数组数计算得到初始 m：$m = t / \log (n)$
<img src="paste_image/2026-01-30-12-24-46.png" width="68%">

**Hyperband**: 实际上使用较多的 HPO 算法
- 在 SH 中，n 代表 exploration，m 代表 exploitation，两者 trade off
- Hyperband 跑多个 SH，每次将 n 变小（保留上一次最好的一半参数），将 m 变大

<img src="paste_image/2026-01-30-12-34-51.png" width="88%">